# IEE575 Lab 2: Gaussian Process Regressor
**Arizona State University — Applied Stochastic Operations Research Models**

---

In Lab 1 we built intuition for multivariate Gaussians, marginalization, and conditioning.  
Today we use those exact tools to implement a Gaussian Process Regressor from scratch.

By the end of this lab you will be able to:
1. Explain what a kernel function does and why it replaces the covariance matrix
2. Implement GP prediction (posterior mean and covariance) using the conditioning formula
3. Tune hyperparameters by maximizing the log marginal likelihood
4. Critically compare the effect of different kernel choices

**Instructions:** Work through each section in order. Every `# YOUR CODE HERE` block must be filled in before you move to the next section. Do not skip the written reflection questions — they carry marks.


## 0. Setup
Run this cell first. Do not modify it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy.optimize import minimize

rng = np.random.default_rng(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print("Setup complete.")


## 1. From Independent Gaussians to Functions

Last week we sampled independent Gaussians at a few x-locations and connected them with lines.  
The result looked jagged because the values at neighbouring x-locations were *independent*.

**The key idea:** if we want smooth functions, nearby points should be *correlated*.  
A **kernel function** $\kappa(x_i, x_j)$ encodes that correlation — it tells us how similar  
the function values at $x_i$ and $x_j$ should be.

### 1a. Independent samples (no kernel)

Run the cell below. This is what you saw last week.


In [ ]:
num_points = 15
num_samples = 8

X = np.linspace(-2, 2, num_points)
samples_independent = rng.standard_normal(size=(num_samples, num_points))

plt.figure(figsize=(7, 3))
for s in samples_independent:
    plt.plot(X, s, '-o', lw=0.8, ms=3, alpha=0.7)
plt.title('Samples from independent Gaussians at each x')
plt.xlabel('x'); plt.ylabel('f(x)')
plt.tight_layout(); plt.show()


**Q1 — Predict before running the next section:**  
If we replace the identity covariance matrix with one built from an RBF kernel,  
will the samples look more or less smooth? Write your prediction here:

> *Your prediction:*


## 2. The RBF Kernel

The **Radial Basis Function (RBF)** kernel, also called the squared exponential kernel, is:

$$\kappa(x_i, x_j) = \sigma^2 \exp\!\left(-\frac{(x_i - x_j)^2}{2\,l^2}\right)$$

- $\sigma^2$ — output variance (controls the amplitude of functions)  
- $l$ — length scale (controls how quickly functions can wiggle)

### 2a. Implement the RBF kernel

Complete the function below. It should return an $(n_1 \times n_2)$ covariance matrix  
where entry $(i,j) = \kappa(x_i, x_j)$.


In [ ]:
def rbf_kernel(X1, X2, sigma=1.0, length_scale=1.0, noise_var=0.0):
    """
    Compute the RBF kernel matrix between X1 and X2.

    Parameters
    ----------
    X1 : array, shape (n1, 1)
    X2 : array, shape (n2, 1)
    sigma        : output scale
    length_scale : length scale l
    noise_var    : noise variance added to diagonal (only when X1 is X_train and X2 is X_train)

    Returns
    -------
    K : array, shape (n1, n2)
    """
    X1 = X1.flatten()
    X2 = X2.flatten()

    # Compute the (n1, n2) matrix of squared differences
    # Hint: np.subtract.outer(X1, X2) gives the (n1, n2) difference matrix
    diff = np.subtract.outer(X1, X2)

    # YOUR CODE HERE
    # K = sigma**2 * exp(- diff**2 / (2 * length_scale**2))
    K = None

    # Add noise to diagonal only when building K(X_train, X_train)
    if noise_var > 0 and X1.shape == X2.shape and np.allclose(X1, X2):
        K += noise_var * np.eye(len(X1))

    return K

# ── Sanity check ──────────────────────────────────────────────────────────────
X_check = np.linspace(-2, 2, 5).reshape(-1, 1)
K_check = rbf_kernel(X_check, X_check, sigma=1.0, length_scale=1.0)
print("Kernel matrix (5x5, sigma=1, l=1):")
print(np.round(K_check, 3))
print()
print("Diagonal all ones?", np.allclose(np.diag(K_check), 1.0))
print("Symmetric?        ", np.allclose(K_check, K_check.T))
print("All entries <= 1? ", np.all(K_check <= 1.0 + 1e-9))


**Q2 — Explain:**  
Why is the diagonal of the kernel matrix always 1 (when $\sigma=1$)?  
What does that mean in terms of the function values at a point?

> *Your answer:*


In [ ]:
K_bad = rbf_kernel(np.array([[0.0], [0.001]]), np.array([[0.0], [0.001]]), length_scale=1.0)

print(np.linalg.cond(K_bad))          # condition number — huge
print(np.linalg.cond(K_bad + 1e-5 * np.eye(2)))   # condition number — fine

### 2b. Effect of length scale

Draw samples from $\mathcal{N}(\mathbf{0}, K)$ where $K$ is built from the RBF kernel  
with four different length scales. Observe how the shape of sampled functions changes.


In [ ]:
N = 200
X_dense = np.linspace(-3, 3, N).reshape(-1, 1)
length_scales = [0.1, 0.5, 1.5, 4.0]

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)

for ax, l in zip(axes.flat, length_scales):
    K = rbf_kernel(X_dense, X_dense, sigma=1.0, length_scale=l, noise_var=1e-6)
    # Draw 5 sample functions from N(0, K)
    samples = rng.multivariate_normal(mean=np.zeros(N), cov=K, size=5)
    for s in samples:
        ax.plot(X_dense.flatten(), s, lw=0.9, alpha=0.8)
    ax.set_title(f'$l = {l}$')
    ax.set_ylim(-3, 3)

fig.supxlabel('x'); fig.supylabel('f(x)')
fig.suptitle('Prior samples from GP with RBF kernel (varying length scale)', y=1.02)
plt.tight_layout(); plt.show()


**Q3 — Explain:**  
Describe in one or two sentences what happens to the sampled functions as $l$ increases.  
Connect your answer back to what the kernel matrix looks like.

> *Your answer:*


## 3. GP Prediction (Posterior Mean and Covariance)

Now we condition on observations. Recall from lecture, the joint prior is:

$$
\begin{pmatrix}\mathbf{f} \\ \mathbf{f}_*\end{pmatrix} \sim \mathcal{N}\!\left(\mathbf{0},\,
\begin{pmatrix}K(X, X) & K(X, X_*) \\ K(X_*, X) & K(X_*, X_*)\end{pmatrix}\right)
$$

Applying the Gaussian conditioning formula gives the posterior:

$$
\mathbf{f}_* \mid \mathbf{f},\, X,\, X_* \;\sim\; \mathcal{N}(\boldsymbol{\mu}_*, \Sigma_*)
$$

$$
\boldsymbol{\mu}_* = K(X_*, X)\, K(X, X)^{-1}\, \mathbf{f}
$$

$$
\Sigma_* = K(X_*, X_*) - K(X_*, X)\, K(X, X)^{-1}\, K(X, X_*)
$$

### 3a. Implement the GP predictor

Fill in the body of `gp_predict`. Map the equations directly to code — each line of math  
should correspond to one or two lines of Python.

**Numerical note:** never call `np.linalg.inv()` to solve a linear system.  
Use `scipy.linalg.solve(A, B)` instead — it is more stable and faster.


In [ ]:
def gp_predict(X_train, y_train, X_test, kernel_func, **kernel_kwargs):
    """
    Compute GP posterior mean and covariance at test points X_test.

    Parameters
    ----------
    X_train : array, shape (n, 1)   — training inputs
    y_train : array, shape (n,)     — training targets
    X_test  : array, shape (m, 1)   — test inputs
    kernel_func                     — callable, returns kernel matrix
    **kernel_kwargs                 — passed to kernel_func (sigma, length_scale, noise_var)

    Returns
    -------
    mu_star  : array, shape (m,)    — posterior mean
    cov_star : array, shape (m, m)  — posterior covariance
    """
    # Step 1: Build the three kernel matrices
    #   K_XX     = K(X_train, X_train)  — includes noise on diagonal
    #   K_XXs    = K(X_train, X_test)   — no noise
    #   K_XsXs   = K(X_test,  X_test)   — no noise

    # YOUR CODE HERE
    K_XX    = None
    K_XXs   = None
    K_XsXs  = None

    # Step 2: Solve K_XX^{-1} K_XXs using scipy.linalg.solve
    # scipy.linalg.solve(A, B) solves A @ x = B for x
    # So K_XX^{-1} K_XXs = solve(K_XX, K_XXs)

    # YOUR CODE HERE
    solved = None   # shape (n, m)

    # Step 3: Posterior mean
    # mu_star = K(X_*, X) K(X,X)^{-1} f  =  solved.T @ y_train

    # YOUR CODE HERE
    mu_star = None

    # Step 4: Posterior covariance
    # cov_star = K(X_*, X_*) - K(X_*, X) K(X,X)^{-1} K(X, X_*)
    #          = K_XsXs - solved.T @ K_XXs

    # YOUR CODE HERE
    cov_star = None

    return mu_star, cov_star


**Q4 — Check your understanding:**  
The posterior covariance formula is $\Sigma_* = K(X_*, X_*) - K(X_*, X)\,K(X,X)^{-1}\,K(X,X_*)$.  
The second term is *subtracted*. What does this mean intuitively?  
What happens to $\Sigma_*$ at a location $x_*$ that is very close to a training point?

> *Your answer:*


### 3b. Run the GP on a test function

In [ ]:
# True function (hidden from the GP — used only for visual comparison)
true_func = lambda x: ((1 - 3*x**2) * np.sin(np.pi*x) * x**2).flatten()

domain = (-1, 1)
X_train = np.array([-0.9, -0.15, 0.35, 0.6, 0.7]).reshape(-1, 1)
y_train = true_func(X_train)
X_test  = np.linspace(domain[0], domain[1], 500).reshape(-1, 1)

# GP hyperparameters (we will tune these in Section 4)
sigma        = 1.0
length_scale = 0.5
noise_var    = 0.01

mu_star, cov_star = gp_predict(
    X_train, y_train, X_test,
    rbf_kernel,
    sigma=sigma, length_scale=length_scale, noise_var=noise_var
)

std_star        = np.sqrt(np.diag(cov_star))
posterior_samples = rng.multivariate_normal(mean=mu_star, cov=cov_star, size=5)


In [ ]:
def plot_gp(X_train, y_train, X_test, mu, std, samples, true_func, title=''):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

    # Top panel: posterior mean + uncertainty band
    ax1.plot(X_test, true_func(X_test), 'b--', lw=1.5, label='True $f(x)$')
    ax1.fill_between(X_test.flatten(), mu - 3*std, mu + 3*std,
                     color='tomato', alpha=0.2, label=r'$\mu \pm 3\sigma$')
    ax1.plot(X_test, mu, 'r-', lw=2, label=r'Posterior mean $\mu_*$')
    ax1.plot(X_train, y_train, 'ko', ms=6, zorder=5, label='Observations')
    ax1.set_ylabel('f(x)'); ax1.legend(loc='upper left', fontsize=9)
    ax1.set_title(title or 'GP Posterior')

    # Bottom panel: sample functions from the posterior
    for s in samples:
        ax2.plot(X_test, s, lw=0.9, alpha=0.7)
    ax2.plot(X_train, y_train, 'ko', ms=6, zorder=5)
    ax2.set_xlabel('x'); ax2.set_ylabel('f(x)')
    ax2.set_title('Posterior sample functions'); ax2.set_xlim(domain)

    plt.tight_layout(); plt.show()

plot_gp(X_train, y_train, X_test, mu_star, std_star, posterior_samples,
        true_func, title=f'GP Posterior  (l={length_scale}, noise_var={noise_var})')


**Q5 — Explain:**  
Where is the uncertainty band widest? Where is it narrowest?  
Is this what you expected? Explain in terms of the conditioning formula.

> *Your answer:*


## 4. Hyperparameter Tuning via Log Marginal Likelihood

We have been hand-picking $l$ and noise variance. In practice we optimise them.

Recall from lecture, the **log marginal likelihood** is:

$$
\log p(\mathbf{y} \mid X, \boldsymbol{\Theta}) =
-\frac{1}{2}\,\mathbf{y}^T K_{\Theta}^{-1} \mathbf{y}
-\frac{1}{2}\log|K_{\Theta}|
-\frac{n}{2}\log(2\pi)
$$

The three terms have interpretations:
- **Data fit:** $-\frac{1}{2}\mathbf{y}^T K^{-1} \mathbf{y}$ — how well the model explains the observations
- **Complexity penalty:** $-\frac{1}{2}\log|K|$ — penalises unnecessarily complex models  
- **Constant:** $-\frac{n}{2}\log(2\pi)$ — does not depend on parameters

### 4a. Implement the negative log marginal likelihood

We minimise the *negative* LML (equivalently maximise the LML).

**Numerical tip:** use `np.linalg.slogdet(K)` to compute $\log|K|$ stably.  
It returns `(sign, logabsdet)` — for a positive definite matrix `sign = 1`,  
so `log|K| = np.linalg.slogdet(K)[1]`.


In [ ]:
def neg_log_marginal_likelihood(params, X_train, y_train):
    """
    Negative log marginal likelihood for GP with RBF kernel.

    Parameters
    ----------
    params  : array [length_scale, log_noise_var]
              (we optimise log_noise_var to keep noise_var > 0)
    X_train : array, shape (n, 1)
    y_train : array, shape (n,)

    Returns
    -------
    nlml : float
    """
    length_scale = params[0]
    noise_var    = np.exp(params[1])   # ensures positivity
    sigma        = 1.0
    n            = len(y_train)

    K = rbf_kernel(X_train, X_train,
                   sigma=sigma, length_scale=length_scale, noise_var=noise_var)

    # Term 1: data fit = 0.5 * y^T K^{-1} y
    # Use scipy.linalg.solve(K, y_train) to compute K^{-1} y_train  (call it alpha)

    # YOUR CODE HERE
    alpha    = None          # K^{-1} y_train,  shape (n,)
    data_fit = None          # 0.5 * y_train @ alpha

    # Term 2: complexity penalty = 0.5 * log|K|
    # Use np.linalg.slogdet(K)[1]  to get log|K|

    # YOUR CODE HERE
    log_det    = None        # log|K|
    complexity = None        # 0.5 * log_det

    # Term 3: normalisation constant = 0.5 * n * log(2*pi)

    # YOUR CODE HERE
    constant = None

    return data_fit + complexity + constant


# Quick test
test_params = [0.5, np.log(0.01)]
nlml_test = neg_log_marginal_likelihood(test_params, X_train, y_train)
print(f"NLML at test params: {nlml_test:.4f}  (should be a finite scalar)")


### 4b. Optimise the hyperparameters

In [ ]:
initial_params = [1.0, np.log(0.1)]   # [length_scale, log_noise_var]

result = minimize(
    neg_log_marginal_likelihood,
    x0=initial_params,
    args=(X_train, y_train),
    method='Nelder-Mead',
    options={'maxiter': 5000, 'xatol': 1e-5, 'fatol': 1e-5}
)

opt_length_scale = result.x[0]
opt_noise_var    = np.exp(result.x[1])

print(f"Optimisation converged : {result.success}")
print(f"Optimised length scale : {opt_length_scale:.4f}")
print(f"Optimised noise variance: {opt_noise_var:.6f}")
print(f"NLML at optimum         : {result.fun:.4f}")


In [ ]:
mu_opt, cov_opt = gp_predict(
    X_train, y_train, X_test,
    rbf_kernel,
    sigma=1.0, length_scale=opt_length_scale, noise_var=opt_noise_var
)
std_opt      = np.sqrt(np.diag(cov_opt))
samples_opt  = rng.multivariate_normal(mean=mu_opt, cov=cov_opt, size=5)

plot_gp(X_train, y_train, X_test, mu_opt, std_opt, samples_opt,
        true_func,
        title=f'GP Posterior — optimised (l={opt_length_scale:.3f}, noise={opt_noise_var:.5f})')


**Q6 — Compare:**  
Compare the GP fit before and after optimisation (Section 3b vs here).  
Did the uncertainty band change? Did the posterior mean improve?  
Refer specifically to the values of $l$ and noise variance that were found.

> *Your answer:*


## 5. Kernel Exploration

The RBF kernel is one of many. Different kernels encode different *prior beliefs* about the  
function — how smooth it is, whether it is periodic, whether it has linear trends.

A comprehensive reference: https://www.cs.toronto.edu/~duvenaud/cookbook/

Below are three additional kernels. Implement them, then run the comparison.

### 5a. Implement three kernels


In [ ]:
def matern52_kernel(X1, X2, sigma=1.0, length_scale=1.0, noise_var=0.0):
    """
    Matern 5/2 kernel:
        kappa(r) = sigma^2 * (1 + sqrt(5)*r/l + 5*r^2/(3*l^2)) * exp(-sqrt(5)*r/l)
    where r = |x_i - x_j|
    """
    X1 = X1.flatten(); X2 = X2.flatten()
    r = np.abs(np.subtract.outer(X1, X2))   # (n1, n2) absolute distances

    # YOUR CODE HERE
    sqrt5 = np.sqrt(5)
    K = None

    if noise_var > 0 and X1.shape == X2.shape and np.allclose(X1, X2):
        K += noise_var * np.eye(len(X1))
    return K


def periodic_kernel(X1, X2, sigma=1.0, length_scale=1.0, period=1.0, noise_var=0.0):
    """
    Periodic kernel:
        kappa(x_i, x_j) = sigma^2 * exp(-2 * sin^2(pi * |x_i - x_j| / p) / l^2)
    """
    X1 = X1.flatten(); X2 = X2.flatten()
    diff = np.abs(np.subtract.outer(X1, X2))

    # YOUR CODE HERE
    K = None

    if noise_var > 0 and X1.shape == X2.shape and np.allclose(X1, X2):
        K += noise_var * np.eye(len(X1))
    return K


def linear_kernel(X1, X2, sigma=1.0, sigma_b=1.0, c=0.0, noise_var=0.0):
    """
    Linear kernel:
        kappa(x_i, x_j) = sigma_b^2 + sigma^2 * (x_i - c)(x_j - c)
    Encodes a prior belief that the function is approximately linear.
    """
    X1 = X1.flatten(); X2 = X2.flatten()

    # YOUR CODE HERE
    K = None

    if noise_var > 0 and X1.shape == X2.shape and np.allclose(X1, X2):
        K += noise_var * np.eye(len(X1))
    return K


### 5b. Compare prior samples across kernels

In [ ]:
N = 200
X_prior = np.linspace(-3, 3, N).reshape(-1, 1)
n_samples = 6

kernels = {
    'RBF (l=0.8)':        (rbf_kernel,      dict(sigma=1.0, length_scale=0.8)),
    'Matern 5/2 (l=0.8)': (matern52_kernel, dict(sigma=1.0, length_scale=0.8)),
    'Periodic (p=1.5)':   (periodic_kernel, dict(sigma=1.0, length_scale=0.8, period=1.5)),
    'Linear':             (linear_kernel,   dict(sigma=1.0, sigma_b=0.5, c=0.0)),
}

fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=True)

for ax, (name, (kfunc, kwargs)) in zip(axes, kernels.items()):
    K = kfunc(X_prior, X_prior, noise_var=1e-6, **kwargs)
    samples = rng.multivariate_normal(np.zeros(N), K, size=n_samples)
    for s in samples:
        ax.plot(X_prior.flatten(), s, lw=0.9, alpha=0.75)
    ax.set_title(name, fontsize=10); ax.set_xlabel('x')

axes[0].set_ylabel('f(x)')
fig.suptitle('Prior samples from four kernels', y=1.02)
plt.tight_layout(); plt.show()


**Q7 — Reflect:**  
For each kernel, describe in one sentence what assumption it encodes about the unknown function.  
Which kernel would you choose if you believed the true function was periodic? Why?

> *RBF:*  
> *Matern 5/2:*  
> *Periodic:*  
> *Linear:*  
> *Choice for periodic data and reason:*


### 5c. Fit all four kernels to the same observations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

for ax, (name, (kfunc, kwargs)) in zip(axes.flat, kernels.items()):
    try:
        mu, cov = gp_predict(X_train, y_train, X_test, kfunc,
                             noise_var=0.01, **kwargs)
        std = np.sqrt(np.diag(cov))
        ax.plot(X_test, true_func(X_test), 'b--', lw=1.2, label='True f(x)')
        ax.fill_between(X_test.flatten(), mu - 2*std, mu + 2*std,
                        alpha=0.25, color='tomato')
        ax.plot(X_test, mu, 'r-', lw=2, label='Posterior mean')
        ax.plot(X_train, y_train, 'ko', ms=6, zorder=5)
        ax.set_title(name); ax.set_xlabel('x'); ax.set_ylabel('f(x)')
        ax.legend(fontsize=8)
    except Exception as e:
        ax.set_title(f'{name}\n(error: {e})')

plt.suptitle('GP posteriors with different kernels (same observations)', y=1.02)
plt.tight_layout(); plt.show()


**Q8 — Reflect:**  
Which kernel fits the data best? Which fits worst?  
Is the best-fitting kernel necessarily the best *model*?  
Think about what happens between observation points and in regions with no data.

> *Your answer:*


## 6. Bonus: Kernel Combination

Kernels can be combined:

- **Sum:** $\kappa_{12}(x_i, x_j) = \kappa_1(x_i, x_j) + \kappa_2(x_i, x_j)$
- **Product:** $\kappa_{12}(x_i, x_j) = \kappa_1(x_i, x_j) \cdot \kappa_2(x_i, x_j)$

**Task:** Create a new kernel combining RBF and Periodic (your choice of sum or product).  
Fit it to the training data and compare to the individual kernels from Section 5c.  
Include a brief explanation of why you chose sum or product.


In [ ]:
# YOUR CODE HERE

def combined_kernel(X1, X2, sigma=1.0, length_scale=0.8, period=1.5, noise_var=0.0):
    """
    Combine RBF and Periodic kernels.
    Explain your choice of combination (sum or product) in this docstring.
    """
    pass


# Fit and plot
# YOUR CODE HERE


---
## Submission Checklist

Before submitting, confirm:

- [ ] All `# YOUR CODE HERE` blocks are filled in
- [ ] All Q1–Q8 reflection questions are answered
- [ ] Section 6 bonus is attempted
- [ ] Notebook runs top to bottom without errors (`Runtime > Run all`)

Submit the `.ipynb` file to Canvas.
